# 10b Seed and Refine the Concept Catalog

Start from the nine seed concepts in the research plan. This is the inspectable, manual counterpart to the proposal stage in 10f. It uses the same development-exclusive examples, example threshold, sample limit, and default LLM settings. This notebook can ask the configured LLM for child-concept hypotheses, but generation is off by default. Review proposed definitions before adding them to the catalog.

Concept-generation examples come only from PMIDs linked exclusively to development pairs, preventing shared papers or confirmation-pair abstracts from entering the prompt. SI values and forecast outcomes are never loaded here.

For one refinement round, first run this notebook with proposal generation off to save the seed catalog, then run 10c and 10d. After reviewing development results, return here, set `PARENT_ID` to a concept you want to refine, and enable `RUN_CONCEPT_PROPOSAL`. Review the proposed children, save the revised catalog, then rerun 10c and 10d. Leave 10e for confirmation after this development round is finished.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from amr.pipeline_config import cfg, load_pipeline_config
from amr.pipelines.concept_notebook_workflows import (
    load_concept_catalog_context, build_development_proposal_examples,
    request_child_concepts, save_child_proposal, save_reviewed_concept_catalog,
)
from amr.si.concepts import SEED_CONCEPTS
PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)
LLM_MODEL = str(cfg(PIPELINE_CONFIG, "llm.model", "gpt-5-nano"))
LLM_TEMPERATURE = 0
LLM_MAX_TOKENS = 4000
NUM_CHILDREN = 3
MAX_EXAMPLES_PER_PARENT = 30
PROPOSAL_EXAMPLE_SEED = 42
POSITIVE_EXAMPLE_THRESHOLD = 0.5
CONTEXT = load_concept_catalog_context(PROJECT_ROOT)
catalog = CONTEXT["catalog"]
print("Catalog version:", CONTEXT["catalog_version"], "concepts:", len(catalog))


## Select development-only examples

The starting definitions are the nine seed concepts documented in the [research plan](../docs/si_concept_discovery_plan.md). The pair split is at the pathogen–antibiotic level. A PMID associated with even one confirmation pair is removed from the proposal examples, so generation prompts cannot learn from confirmation-linked articles.

Examples are drawn from abstracts already scored as present for the selected parent. The `POSITIVE_EXAMPLE_THRESHOLD`, `MAX_EXAMPLES_PER_PARENT`, and `PROPOSAL_EXAMPLE_SEED` controls match notebook 10f. If none are available, first classify the catalog in notebook 10c. Inspect the examples before requesting a proposal.

In [ ]:
PARENT_ID = "surveillance"
EXAMPLES = build_development_proposal_examples(
    PROJECT_ROOT, parent_id=PARENT_ID, positive_threshold=POSITIVE_EXAMPLE_THRESHOLD,
    max_examples=MAX_EXAMPLES_PER_PARENT, random_seed=PROPOSAL_EXAMPLE_SEED,
)
examples = EXAMPLES["examples"]
print(f"Development-only examples for {PARENT_ID}: {len(examples):,}")
if not examples:
    print("Run notebook 10c on the current parent concept first; refinement needs scored parent examples.")
display(EXAMPLES["examples_df"].drop(columns=["abstract"], errors="ignore"))


## Optional child-concept proposal
The manual proposal path exposes the same `NUM_CHILDREN`, model, temperature, and token-limit controls used by 10f.


A proposal is a hypothesis for a useful refinement, not an accepted label. Check whether suggested children can be distinguished from the parent and one another. Refine definitions and inclusion/exclusion criteria so the boundaries can be applied from title and abstract alone. Reject suggestions that rely on information unavailable in those texts.

In [ ]:
RUN_CONCEPT_PROPOSAL = False
if RUN_CONCEPT_PROPOSAL:
    if not examples:
        raise RuntimeError("No development-only scored examples for the parent; classify the current catalog first.")
    parent = next(concept for concept in catalog if concept["concept_id"] == PARENT_ID)
    PROPOSAL = request_child_concepts(
        parent, examples, model_name=LLM_MODEL, temperature=LLM_TEMPERATURE,
        max_tokens=LLM_MAX_TOKENS, num_children=NUM_CHILDREN,
    )
    PROPOSAL_PATH = save_child_proposal(PROJECT_ROOT, PARENT_ID, PROPOSAL)
    print("Review proposal before editing catalog:", PROPOSAL_PATH)
else:
    print("LLM proposal disabled. Set RUN_CONCEPT_PROPOSAL=True to request hypotheses.")


## Save the reviewed catalog

Edit the catalog only after reviewing any proposal. Every concept needs a stable `concept_id`, definition, and inclusion/exclusion criteria; child concepts also need a `parent_id`. Saving writes the current catalog and a hash-named snapshot. Notebook 10c uses that hash to keep scores for different catalog versions separate.

In [ ]:
# Add reviewed child dictionaries here. Each needs a stable concept_id, parent_id,
# definition, inclusion_criteria, and exclusion_criteria.
REVIEWED_CHILDREN = []
catalog.extend(REVIEWED_CHILDREN)
SAVED = save_reviewed_concept_catalog(PROJECT_ROOT, catalog)
print("Saved catalog:", SAVED["catalog_path"])
print("Saved immutable version:", SAVED["snapshot_path"])
print("Catalog version:", SAVED["catalog_version"])
